# 09 - ESMFold per-variant delta features

Folds all 4,770 variants (wet-lab-sealed mutations excluded), extracts per-variant absolute and
delta structural features, writes `esmfold_variant_features.csv` into `data/interim/` keyed by
`mutant`. If you're running this inside the full project, drop that file into the real
`data/interim/` and re-run notebook 06 to add the full-coverage delta features to the
predicted-structure tier.

## This folder is self-contained -- send the whole `colab_esmfold/` folder, not just this file

Everything this notebook needs to run travels with this folder:

| what | why it's here |
|---|---|
| `.projectroot` | marker this notebook's path-detection looks for -- lets it find its own root without any absolute paths |
| `paths.py` | defines RAW/INTERIM/PROCESSED/etc. relative to whatever folder holds `.projectroot` |
| `data/processed/tem1_firnberg_processed.parquet` | the canonical 4,783-row variant table (13 wet-lab-sealed rows included but flagged), the only real data dependency -- everything else (`wt_seq.json`, the variant manifest) is derived from it in the first code cell |

The first code cell also installs its own Python dependencies (`torch>=2.4`, `transformers`,
`accelerate`, `biotite`, `scipy`, `pandas`) into whatever kernel runs it, no pre-provisioned conda
env assumed. Restart the kernel after that cell runs once if this is the first time
torch/transformers get installed in this environment, since transformers caches its
PyTorch-availability check at import time and a same-kernel re-run won't pick up a fresh install.

To run on Colab: upload this whole `colab_esmfold/` folder (zip it, upload, unzip in Colab, or
put it in Google Drive and `drive.mount()`), `%cd` into it, then run the notebook top to bottom.
Nothing here calls the `google.colab` API, so no code changes are needed either way.

A GPU is strongly recommended, since 4,770 folds on CPU is very slow. CUDA and Apple-Silicon
MPS both auto-detect, and CPU is a working fallback.

One thing worth knowing if you also have the full project checked out locally: because this
folder carries its own `.projectroot`, running this notebook from inside the full project will
still resolve to this local copy first (the path search stops at the nearest marker going up),
not the outer project's `data/`. That's the point for portability, but it means this bundled
parquet copy won't automatically pick up changes if the outer project's canonical table gets
rebuilt later, so re-copy it here if that happens.

The two files `colab_variant_manifest.csv` and `colab_wt_spec.json` in this folder are earlier,
now-unused inputs. This notebook derives both directly from the parquet above; they're kept for
reference, not read by the current code.


In [1]:
# installs into whichever kernel/env is actually running this notebook (uses the %pip magic,
# not !pip, so it targets the live kernel rather than a possibly-different environment).
# torch pinned >=2.4 because transformers refuses to enable PyTorch models at all below that
# (silently falls back to a torch-less dummy backend and EsmForProteinFolding raises ImportError).
%pip install -q "torch>=2.4" transformers accelerate biotite scipy pandas
print ('done')

Note: you may need to restart the kernel to use updated packages.
done


In [2]:
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".projectroot").exists())
sys.path.insert(0, str(root))
from paths import INTERIM, PROCESSED

import torch, numpy as np, pandas as pd, json, time, os
from transformers import EsmForProteinFolding, AutoTokenizer
import biotite.structure as struc, biotite.structure.io.pdb as pdb
from scipy.spatial.distance import cdist
from io import StringIO

print("project root:", root)
print("torch:", torch.__version__)
print ('done')

project root: /Users/kdave2/Beta-Lactamase Mutagenesis/Beta Lactam ML v2
torch: 2.13.0
done


In [ ]:
# load the canonical variant table first -- it's the actual source of truth (produced by
# notebook 02), and everything below is derived from it, not assumed to pre-exist
canon = pd.read_parquet(PROCESSED / "tem1_firnberg_processed.parquet")
man = canon.loc[~canon["excluded_wetlab_validation"], ["mutant", "position_linear", "mut_aa"]].reset_index(drop=True)
assert len(man) == 4770, f"expected 4770 non-sealed variants, got {len(man)}"

# wt_seq.json is supposed to be written by notebook 03 -- rebuild it here if that hasn't been run
# yet, so this notebook doesn't have a fragile hidden dependency on notebook execution order. Also
# write it back to data/interim so fold_wt.py / fold_variants.py (which expect it there too) stop
# being blocked by the same gap.
wt_spec_path = INTERIM / "wt_seq.json"
if wt_spec_path.exists():
    spec = json.load(open(wt_spec_path))
    wt, positions = spec["wt_seq"], spec["positions"]
else:
    pool = canon[~canon["excluded_wetlab_validation"]]
    positions = sorted(pool["position_linear"].unique().tolist())
    wt_by_pos = {}
    for _, r in pool.iterrows():
        wt_by_pos.setdefault(r["position_linear"], r["wt_aa"])
    wt = "".join(wt_by_pos[p] for p in positions)
    assert len(wt) == 263 and wt.startswith("HPETLVK"), f"reconstructed WT looks wrong: {wt[:10]}"
    json.dump({"wt_seq": wt, "positions": positions}, open(wt_spec_path, "w"))
    print("wt_seq.json was missing -- rebuilt it from the canonical table and wrote it to", wt_spec_path)

pos2idx = {p: i for i, p in enumerate(positions)}
L = len(wt)
print(f"WT len {L}, variants {len(man)}")

# CUDA -> MPS -> CPU, same priority order used everywhere else in this project
if torch.cuda.is_available():
    dev = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    dev = "mps"
else:
    dev = "cpu"

tok = AutoTokenizer.from_pretrained("facebook/esmfold_v1")
model = EsmForProteinFolding.from_pretrained("facebook/esmfold_v1", low_cpu_mem_usage=True)
model = model.to(dev).eval()
if dev == "cuda":
    model.esm = model.esm.half()  # fp16 for speed/memory; skip on mps (unreliable half support) and cpu
model.trunk.set_chunk_size(64)
print("device:", dev)


In [ ]:
MAXASA={'A':129,'R':274,'N':195,'D':193,'C':167,'E':223,'Q':225,'G':104,'H':224,'I':197,
        'L':201,'K':236,'M':224,'F':240,'P':159,'S':155,'T':172,'W':285,'Y':263,'V':174}
def struct_feats(pdb_str, idx, aa):
    f=pdb.PDBFile.read(StringIO(pdb_str)); arr=f.get_structure(model=1)
    arr=arr[struc.filter_amino_acids(arr)]
    sasa=struc.apply_residue_wise(arr, struc.sasa(arr,vdw_radii="Single"), np.nansum)
    rel=sasa[idx]/MAXASA.get(aa,200)
    ca=arr[arr.atom_name=="CA"]; Dm=cdist(ca.coord,ca.coord)
    cn=((Dm[idx]<8.0)&(Dm[idx]>0)).sum()
    lo,hi=max(0,idx-2),min(L,idx+3); cn-=((Dm[idx,lo:hi]<8.0)&(Dm[idx,lo:hi]>0)).sum()
    return float(sasa[idx]), float(rel), 1.0-min(max(rel,0),1), float(cn)

def fold_seq(seq):
    inp=tok([seq],return_tensors="pt",add_special_tokens=False)["input_ids"].to(dev)
    with torch.no_grad(): out=model(inp)
    plddt=out["plddt"][0,:,1].cpu().numpy()*100.0   # per-residue CA pLDDT (0-100)
    return out, plddt

In [ ]:
wt_out, wt_plddt = fold_seq(wt)
wt_pdb = model.output_to_pdb(wt_out)[0]
(INTERIM / "esmfold_wt_variant_run.pdb").write_text(wt_pdb)
wt_feat = {}
for p in positions:
    i = pos2idx[p]; s, rel, bur, cn = struct_feats(wt_pdb, i, wt[i])
    wt_feat[p] = dict(plddt=wt_plddt[i], sasa=s, rel_sasa=rel, burial=bur, contact=cn)
print("WT folded, mean pLDDT %.1f" % wt_plddt.mean())


In [ ]:
CK = INTERIM / "esmfold_variant_features.csv"
rows = pd.read_csv(CK).to_dict("records") if CK.exists() else []
done = {r["mutant"] for r in rows}
t0 = time.time()
for n, (_, r) in enumerate(man.iterrows()):
    if r.mutant in done:
        continue
    i = pos2idx[int(r.position_linear)]; m = r.mut_aa
    s = list(wt); s[i] = m; out, plddt = fold_seq("".join(s))
    pdbs = model.output_to_pdb(out)[0]
    sasa, rel, bur, cn = struct_feats(pdbs, i, m)
    w = wt_feat[int(r.position_linear)]
    rows.append(dict(mutant=r.mutant,
        v_plddt=plddt[i], v_sasa=sasa, v_rel_sasa=rel, v_burial=bur, v_contact=cn,
        d_plddt=plddt[i]-w["plddt"], d_sasa=sasa-w["sasa"], d_rel_sasa=rel-w["rel_sasa"],
        d_burial=bur-w["burial"], d_contact=cn-w["contact"]))
    if len(rows) % 25 == 0 or n == len(man) - 1:
        pd.DataFrame(rows).to_csv(CK, index=False)
        el = time.time() - t0; rate = (len(rows) - len(done)) / max(el, 1e-9)
        print(f"{len(rows)}/{len(man)} {el/60:.1f}min {rate:.2f}/s ETA {(len(man)-len(rows))/max(rate,1e-9)/60:.0f}min")
pd.DataFrame(rows).to_csv(CK, index=False)
print("DONE ->", CK)


In [ ]:
# already written to data/interim -- no download step needed locally; drop straight into
# notebook 06 to add the full-coverage delta features to the predicted-structure tier
print("wt fold  ->", INTERIM / "esmfold_wt_variant_run.pdb")
print("features ->", INTERIM / "esmfold_variant_features.csv")
